# HDAT DS 독립 종합문제 — 차량 센서 시계열 (Solution)

> 현대자동차그룹 또는 현대엔지비의 공식·복원 문제가 아닌 **독립 창작 연습문제**입니다. 외부 데이터 없이 합성 데이터를 만들고, 모델링은 PyTorch만 사용합니다.

과거 24시점의 차량 센서로 다음 시점의 위험 여부를 예측합니다. 시간 분할부터 저장된 모델로 submission을 재생성하는 단계까지 한 가지 안전한 참고 구현을 제공합니다.

## 환경과 운영 습관

- 권장: Python 3.10–3.12, PyTorch 2.2 이상, JupyterLab/Notebook 7 이상
- CPU 전용, 약 900시점과 6 epoch라 일반 노트북에서 빠르게 끝납니다.
- 시작 전 **Kernel → Restart Kernel and Clear Outputs**, 각 큰 단계 후 **Ctrl/Cmd+S**를 누르세요.
- 제출 전에는 Restart + Run All로 실행 순서 의존성을 없애고, CSV를 다시 읽어 행 수·열 이름·NaN을 확인하세요.
- 해설 코드는 유일한 정답이 아닙니다. 핵심은 각 단계의 데이터 계약과 누수 방지 이유를 설명할 수 있는 것입니다.

In [ ]:
import csv
import platform
import random
import tempfile
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print('Python :', platform.python_version(), '(권장 3.10–3.12)')
print('PyTorch:', torch.__version__, '(권장 2.2+)')
SEED = 2026
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cpu')

## 데이터 명세

특징 순서는 `speed`, `rpm`, `coolant`, `vibration`, `throttle`입니다. 레이블 1은 해당 시점의 합성 위험 신호가 train 구간 기준 상위 수준이라는 뜻입니다. 실제 차량 진단 기준이 아니며 학습용으로만 사용합니다.

In [ ]:
def make_vehicle_sensor_series(n_steps=900, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    t = torch.arange(n_steps, dtype=torch.float32)

    def noise(scale):
        return scale * torch.randn(n_steps, generator=g)

    throttle = (0.45 + 0.24 * torch.sin(t / 19) + noise(0.06)).clamp(0, 1)
    speed = (54 + 17 * torch.sin(t / 31) + 8 * throttle + noise(2.5)).clamp_min(0)
    rpm = 650 + 27 * speed + 720 * throttle + noise(90)
    coolant = 79 + 0.012 * t + 4 * torch.sin(t / 83) + 2.5 * throttle + noise(0.8)
    vibration = 0.18 + 0.00016 * rpm + 0.09 * torch.relu(torch.sin(t / 11)) + noise(0.025)
    features = torch.stack([speed, rpm, coolant, vibration, throttle], dim=1).to(torch.float32)

    risk = (
        0.10 * (coolant - 82)
        + 2.4 * (vibration - 0.45)
        + 0.00045 * (rpm - 2100)
        + 0.7 * throttle
        + noise(0.35)
    )
    train_boundary = int(0.60 * n_steps)
    threshold = torch.quantile(risk[:train_boundary], 0.72)
    labels = (risk > threshold).to(torch.float32)
    return features, labels

feature_names = ['speed', 'rpm', 'coolant', 'vibration', 'throttle']
raw_X, raw_y = make_vehicle_sensor_series()
assert raw_X.shape == (900, 5) and raw_y.shape == (900,)
assert raw_X.dtype == torch.float32 and raw_y.dtype == torch.float32
assert torch.isfinite(raw_X).all() and torch.isfinite(raw_y).all()
print('raw:', raw_X.shape, raw_y.shape, 'positive rate=', round(raw_y.mean().item(), 3))

## 1. 시간 순서 분할

시계열에서 무작위 분할은 미래 패턴을 train에 섞을 수 있습니다. 명시적인 두 경계로 과거→미래 순서를 유지합니다.

In [ ]:
n_steps = raw_X.shape[0]
train_end = int(0.60 * n_steps)
val_end = int(0.80 * n_steps)
X_train_raw, y_train_raw = raw_X[:train_end], raw_y[:train_end]
X_val_raw, y_val_raw = raw_X[train_end:val_end], raw_y[train_end:val_end]
X_test_raw, y_test_raw = raw_X[val_end:], raw_y[val_end:]

assert X_train_raw.shape == (540, 5) and y_train_raw.shape == (540,)
assert X_val_raw.shape == (180, 5) and y_val_raw.shape == (180,)
assert X_test_raw.shape == (180, 5) and y_test_raw.shape == (180,)
assert torch.equal(X_val_raw[0], raw_X[540])
print('split:', len(X_train_raw), len(X_val_raw), len(X_test_raw))

## 2. train-only 표준화

validation/test의 평균까지 사용하면 미래 분포를 미리 본 데이터 누수입니다. train 통계는 체크포인트에도 함께 저장해야 추론을 재현할 수 있습니다.

In [ ]:
train_mean = X_train_raw.mean(dim=0, keepdim=True)
train_std = X_train_raw.std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)
X_train_n = (X_train_raw - train_mean) / train_std
X_val_n = (X_val_raw - train_mean) / train_std
X_test_n = (X_test_raw - train_mean) / train_std

assert train_mean.shape == (1, 5) and train_std.shape == (1, 5)
assert X_train_n.shape == (540, 5) and X_val_n.shape == (180, 5)
assert torch.all(train_std > 0)
assert torch.allclose(X_train_n.mean(0), torch.zeros(5), atol=1e-5)

## 3. 시계열 window 만들기

첫 window는 0–23 시점을 입력으로 보고 24 시점의 레이블을 예측합니다. split마다 별도로 호출하므로 어떤 window도 경계를 넘지 않습니다.

In [ ]:
WINDOW = 24
HORIZON = 1

def make_windows(features, labels, window=WINDOW, horizon=HORIZON):
    n_samples = len(features) - window - horizon + 1
    if n_samples <= 0:
        raise ValueError('window와 horizon이 split 길이보다 큽니다')
    windows = torch.stack([features[i:i + window] for i in range(n_samples)])
    target_idx = torch.arange(n_samples) + window + horizon - 1
    targets = labels[target_idx].unsqueeze(1)
    return windows.contiguous(), targets.contiguous()

X_train, y_train = make_windows(X_train_n, y_train_raw)
X_val, y_val = make_windows(X_val_n, y_val_raw)
X_test, y_test = make_windows(X_test_n, y_test_raw)

assert X_train.shape == (516, 24, 5) and y_train.shape == (516, 1)
assert X_val.shape == (156, 24, 5) and y_val.shape == (156, 1)
assert X_test.shape == (156, 24, 5) and y_test.shape == (156, 1)
assert X_train.dtype == torch.float32 and y_train.dtype == torch.float32
assert y_train[0].item() == y_train_raw[WINDOW].item()
print('windowed:', X_train.shape, X_val.shape, X_test.shape)

## 4. Majority baseline

다수 class만 예측하는 가장 단순한 기준입니다. 불균형 데이터에서는 accuracy만 높을 수 있으므로 뒤에서 precision/recall/F1도 함께 봅니다.

In [ ]:
train_positive_rate = y_train.mean().item()
majority_class = float(train_positive_rate >= 0.5)
baseline_pred = torch.full_like(y_val, majority_class)
baseline_val_accuracy = (baseline_pred == y_val).float().mean().item()

assert majority_class in (0.0, 1.0)
assert isinstance(baseline_val_accuracy, float)
assert 0.0 <= baseline_val_accuracy <= 1.0
print(f'train positive rate: {train_positive_rate:.3f}')
print(f'baseline val accuracy: {baseline_val_accuracy:.3f}')

## 5. 작은 PyTorch GRU

GRU가 반환하는 `h_n[-1]`은 마지막 layer의 요약 상태 `[B, hidden]`입니다. 이를 Linear에 넣어 logit 하나를 만듭니다.

In [ ]:
class VehicleRiskGRU(nn.Module):
    def __init__(self, n_features=5, hidden_size=16):
        super().__init__()
        self.gru = nn.GRU(n_features, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.gru(x)
        return self.head(h_n[-1])

model = VehicleRiskGRU().to(device)
sample_logits = model(torch.zeros(4, WINDOW, 5))
assert sample_logits.shape == (4, 1) and sample_logits.dtype == torch.float32
print('parameters:', sum(p.numel() for p in model.parameters()))

## 6. DataLoader와 학습

train만 shuffle합니다. 양성 class가 적어 `pos_weight = negative / positive`를 사용하지만, 실제 문제에서는 지표와 분포를 보고 선택해야 합니다.

In [ ]:
train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)
loader_g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=loader_g)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

n_positive = y_train.sum().clamp_min(1.0)
n_negative = y_train.numel() - y_train.sum()
pos_weight = (n_negative / n_positive).reshape(1)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

train_losses = []
for epoch in range(1, 7):
    model.train()
    loss_sum = 0.0
    item_count = 0
    for xb, yb in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        loss_sum += loss.item() * len(xb)
        item_count += len(xb)
    train_losses.append(float(loss_sum / item_count))

assert len(train_losses) == 6
assert all(isinstance(v, float) and torch.isfinite(torch.tensor(v)) for v in train_losses)
print('train losses:', [round(v, 4) for v in train_losses])

## 7. Validation metrics

확률 0.5는 logit 0과 같습니다. 실제 시험에서는 요구 지표를 우선하고, threshold 튜닝을 한다면 validation에서만 결정한 뒤 test에는 한 번 적용합니다.

In [ ]:
def evaluate_binary(model, X, y):
    model.eval()
    with torch.inference_mode():
        pred = (model(X) >= 0).to(y.dtype)
    tp = ((pred == 1) & (y == 1)).sum().item()
    fp = ((pred == 1) & (y == 0)).sum().item()
    fn = ((pred == 0) & (y == 1)).sum().item()
    correct = (pred == y).sum().item()
    eps = 1e-12
    accuracy = correct / y.numel()
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    return {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
    }

val_metrics = evaluate_binary(model, X_val, y_val)
assert set(val_metrics) == {'accuracy', 'precision', 'recall', 'f1'}
assert all(isinstance(v, float) and 0.0 <= v <= 1.0 for v in val_metrics.values())
print('baseline accuracy:', round(baseline_val_accuracy, 3))
print('model metrics    :', {k: round(v, 3) for k, v in val_metrics.items()})

## 8. 체크포인트 재로딩과 submission 검증

체크포인트에는 추론 재현에 필요한 모델 파라미터와 전처리 설정을 함께 넣습니다. test 레이블은 합성 데이터 점검용으로 존재하지만 submission 생성에는 사용하지 않습니다.

In [ ]:
tmp_dir = tempfile.TemporaryDirectory()
checkpoint_path = Path(tmp_dir.name) / 'vehicle_gru.pt'
submission_path = Path(tmp_dir.name) / 'submission.csv'

torch.save(
    {
        'model_state': model.state_dict(),
        'n_features': len(feature_names),
        'hidden_size': 16,
        'feature_names': feature_names,
        'train_mean': train_mean,
        'train_std': train_std,
        'window': WINDOW,
        'horizon': HORIZON,
    },
    checkpoint_path,
)

try:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
except TypeError:  # PyTorch 구버전 호환
    checkpoint = torch.load(checkpoint_path, map_location='cpu')

assert checkpoint['feature_names'] == feature_names
assert checkpoint['window'] == WINDOW and checkpoint['horizon'] == HORIZON
reloaded_model = VehicleRiskGRU(
    n_features=checkpoint['n_features'],
    hidden_size=checkpoint['hidden_size'],
)
reloaded_model.load_state_dict(checkpoint['model_state'])
reloaded_model.eval()
with torch.inference_mode():
    test_probability = torch.sigmoid(reloaded_model(X_test))

with submission_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['sample_id', 'risk_probability'])
    writer.writeheader()
    for sample_id, probability in enumerate(test_probability.squeeze(1).tolist()):
        writer.writerow({'sample_id': sample_id, 'risk_probability': f'{probability:.8f}'})

with submission_path.open('r', newline='', encoding='utf-8') as f:
    reloaded_rows = list(csv.DictReader(f))

assert checkpoint_path.exists() and submission_path.exists()
assert isinstance(reloaded_model, VehicleRiskGRU)
assert test_probability.shape == (156, 1)
assert torch.isfinite(test_probability).all()
assert torch.all((0 <= test_probability) & (test_probability <= 1))
assert len(reloaded_rows) == 156
assert set(reloaded_rows[0]) == {'sample_id', 'risk_probability'}
assert all(row['risk_probability'] != '' for row in reloaded_rows)
print('submission contract OK:', submission_path)
print('first rows:', reloaded_rows[:3])

## 제출 전 체크리스트

- [x] 시간 순서를 지켰고 window가 split 경계를 넘지 않는다.
- [x] 평균·표준편차는 train만으로 계산했다.
- [x] baseline과 모델 validation 지표를 함께 기록했다.
- [x] `eval` + inference 문맥으로 예측했다.
- [x] 전처리 설정과 `state_dict`를 저장해 새 모델로 재로딩했다.
- [x] CSV를 재로딩해 열 이름, 행 수, 확률 범위를 확인했다.
- [ ] 실제 사용 전 Kernel Restart + Run All 후 Ctrl/Cmd+S 한다.